# Monitoring Notebook

This notebook builds the monthly brand monitoring table from Gold data.

## Outputs
- `gold_layer.brand_monitoring` — monthly brand revenue, market share, previous-month share, and change in percentage points
- Zero-sales months are filled so a brand can't disappear from monitoring
- `is_complete_month` flag marks months that are fully captured (all except the latest)

## Alert Rule
> Alert when Brand#32's share in any category falls by at least 5 percentage points compared with the previous complete month.

The alert query is saved in `sql/brand_share_alert.sql`.

In [0]:
%sql
CREATE OR REPLACE TABLE bda_gold.brand_monitoring AS
WITH
brand_categories AS (
    SELECT DISTINCT brand, category FROM bda_gold.brand_sales
),
months AS (
    SELECT DISTINCT month_start FROM bda_gold.brand_sales
),
calendar AS (
    SELECT m.month_start, bc.brand, bc.category
    FROM months m
    CROSS JOIN brand_categories bc
),
actuals AS (
    SELECT month_start, category, brand, SUM(net_revenue) AS brand_revenue
    FROM bda_gold.brand_sales
    GROUP BY month_start, category, brand
),
filled AS (
    SELECT
        c.month_start, c.brand, c.category,
        COALESCE(a.brand_revenue, 0) AS brand_revenue
    FROM calendar c
    LEFT JOIN actuals a
        ON c.month_start = a.month_start
        AND c.brand = a.brand
        AND c.category = a.category
),
with_totals AS (
    SELECT
        f.month_start, f.brand, f.category, f.brand_revenue,
        SUM(f.brand_revenue) OVER (PARTITION BY f.month_start, f.category) AS category_revenue,
        f.brand_revenue / NULLIF(
            SUM(f.brand_revenue) OVER (PARTITION BY f.month_start, f.category), 0
        ) AS market_share
    FROM filled f
),
with_lag AS (
    SELECT
        *,
        LAG(market_share) OVER (
            PARTITION BY brand, category ORDER BY month_start
        ) AS previous_market_share
    FROM with_totals
)
SELECT
    month_start,
    brand,
    category,
    brand_revenue,
    category_revenue,
    market_share,
    previous_market_share,
    (market_share - previous_market_share) * 100 AS share_change_percentage_points,
    CASE WHEN month_start < (
        SELECT MAX(month_start) FROM bda_gold.brand_sales
    ) THEN true ELSE false END AS is_complete_month
FROM with_lag

In [0]:
%sql
-- Show Brand#32 monitoring data (latest months)
SELECT
    month_start,
    category,
    ROUND(brand_revenue, 2) AS brand_revenue,
    ROUND(category_revenue, 2) AS category_revenue,
    ROUND(market_share * 100, 2) AS market_share_pct,
    ROUND(previous_market_share * 100, 2) AS prev_share_pct,
    ROUND(share_change_percentage_points, 2) AS share_change_pp,
    is_complete_month
FROM bda_gold.brand_monitoring
WHERE brand = 'Brand#32'
ORDER BY month_start DESC, category
LIMIT 20;

In [0]:
%sql
-- Show the latest complete month's Brand#32 shares
SELECT
    month_start,
    category,
    ROUND(market_share * 100, 2) AS market_share_pct,
    ROUND(share_change_percentage_points, 2) AS share_change_pp,
    is_complete_month
FROM bda_gold.brand_monitoring
WHERE brand = 'Brand#32'
  AND is_complete_month
  AND month_start = (
      SELECT MAX(month_start)
      FROM bda_gold.brand_monitoring
      WHERE is_complete_month
  )
ORDER BY category;

## Alert Demo

The real Gold tables are historical and unlikely to trigger the alert.
Below we create a **synthetic** monitoring result with a 6-percentage-point drop for Brand#32 to demonstrate the alert trigger.

**Alert rule:** Trigger when Brand#32's share in any category falls by at least 5 percentage points compared with the previous complete month.

The production alert query is in `sql/brand_share_alert.sql`.

In [0]:
# Create a synthetic monitoring result with a 6pp drop for Brand#32
# This simulates what the alert would catch in production

# Get the latest two complete months for Brand#32
latest_months = spark.sql("""
    SELECT DISTINCT month_start
    FROM bda_gold.brand_monitoring
    WHERE is_complete_month
    ORDER BY month_start DESC
    LIMIT 2
""").collect()

if len(latest_months) >= 2:
    latest_month = latest_months[0][0]
    prev_month = latest_months[1][0]

    # Build synthetic data: copy real data but reduce Brand#32 share by 6pp in one category
    demo_df = spark.sql(f"""
        SELECT
            month_start, brand, category,
            brand_revenue, category_revenue,
            market_share,
            previous_market_share,
            -- Simulate 6pp drop for Brand#32 in the latest month
            CASE
                WHEN brand = 'Brand#32' AND month_start = '{latest_month}'
                THEN (market_share - 0.06) * 100
                ELSE (market_share - COALESCE(previous_market_share, market_share)) * 100
            END AS share_change_percentage_points,
            is_complete_month
        FROM bda_gold.brand_monitoring
        WHERE brand = 'Brand#32'
    """)

    demo_df.createOrReplaceTempView("demo_brand_monitoring")

    print(f"Synthetic demo created: Brand#32 share dropped 6pp in latest month ({latest_month})")
    print(f"Previous month for comparison: {prev_month}")

    # Show the demo data
    print("\nDemo monitoring data for Brand#32 (latest month):")
    spark.sql(f"""
        SELECT month_start, category,
               ROUND(market_share * 100, 2) AS share_pct,
               ROUND(share_change_percentage_points, 2) AS change_pp
        FROM demo_brand_monitoring
        WHERE month_start = '{latest_month}'
        ORDER BY category
    """).show(truncate=False)
else:
    print("Not enough complete months for demo")

In [0]:
%sql
-- Run the alert query against the SYNTHETIC demo data
-- (the real alert query in sql/brand_share_alert.sql targets gold_layer.brand_monitoring)
WITH latest AS (
    SELECT MAX(month_start) AS month_start
    FROM demo_brand_monitoring
    WHERE is_complete_month
)
SELECT COUNT(*) AS affected_category_count
FROM demo_brand_monitoring m
JOIN latest
    ON m.month_start = latest.month_start
WHERE m.brand = 'Brand#32'
  AND m.share_change_percentage_points <= -5;